In [1]:
# =============================================================================
# CELL 0: Imports
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings

In [2]:
# =============================================================================
# CELL 1: Configuration and Setup (Economic Indicators Data)
# =============================================================================

NOTEBOOK_NAME = "06: Economic Indicators Data Ingestion"
DATA_SUBFOLDER = "economy"
OBJECTIVE = """Process Statistik Austria economic indicators (monthly)
18 economic variables including production indices, trade, employment
Date range: 2009-2025

IMPORTANT ARCHITECTURAL DECISION:
Economic data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate, production, and gas data architecture.
Economic variables can only be used in monthly-level analyses.

DATE FORMAT: mmm.yy (e.g., Jän.09) with Austrian German month abbreviations
COMPLEX STRUCTURE: Multiple header rows and footer metadata require careful skipping."""

# Economic data specific configuration
ECON_FILE_NAME = "table_2025-09-16_13-30-40.xlsx"
DATE_COLUMN = "Berichtszeitraum"
SKIPROWS = list(range(9)) + [10, 11]  # Skip rows 0-8 (metadata), 10-11 (wertangabe)
NROWS = 213  # Stop at row 213 (metadata text follows)
USECOLS = list(range(1, 20))  # Skip column 0 (empty), use columns 1-19

# Austrian German month abbreviations mapping (extended from gas prices)
GERMAN_MONTHS = {
    'Jän': '01', 'Jan': '01', 'Feb': '02', 'Mär': '03', 'Mrz': '03',
    'Apr': '04', 'Mai': '05', 'Jun': '06', 'Jul': '07',
    'Aug': '08', 'Sep': '09', 'Okt': '10', 'Nov': '11', 'Dez': '12'
}

# Column mapping: Source column name → Target column name
COLUMN_MAPPING = {
    'Berichtszeitraum': 'date',
    'Produktionsindex Industrie (at; 2021=100)': 'econ_prod_index_industry',
    'Verbraucherpreisindex (2015=100)': 'econ_consumer_price_index',
    'Ausfuhren Insgesamt in €': 'econ_exports_total_EUR',
    'Umsatzindex Handel (nom.; 2021=100)': 'econ_turnover_index_commercial_sales',
    'Nächtigungen': 'econ_count_overnight_stays',
    'Umsatz Industrie inTsd.€ (KJE)': 'econ_turnover_industry',
    'Beschäftigte Industrie gesamt (KJE)': 'econ_count_employees_industry',
    'Produktionsindex Bau (at; 2021=100)': 'econ_prod_index_construction',
    'Technische Gesamtproduktion Bau in Tsd. € (KJE)': 'econ_total_production_construction',
    'Umsatzindex Bau (2021=100)': 'econ_turnover_index_construction',
    'Umsatz Bau in Tsd. € (KJE)': 'econ_turnover_construction',
    'Beschäftigte Bau gesamt (KJE)': 'econ_count_employees_construction',
    'Umsatzindex - Großhandel (G46; nom.; 2021=100)': 'econ_turnover_index_wholesale',
    'Umsatzindex - Einzelhandel (G47; nom.; 2021=100)': 'econ_turnover_index_retail',
    'Beschäftigtenindex Handel  (2021=100)': 'econ_count_employees_retail',
    'Umsatzindex - KfZ-Handel (G45; nom.; 2021=100)': 'econ_turnover_index_car_retail',
    'Einfuhren Insgesamt in €': 'econ_imports_total_EUR',
    'Einfuhren Insg. SITC-3 Brennstoffe, Energie in €': 'econ_imports_energy_EUR'
}

# Setup functions (reused from previous pipelines - consistent)
def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """Set up project directory paths for any data source."""
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# Setup execution
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

print("Cell 1: Configuration and Setup - READY")

NOTEBOOK: 06: Economic Indicators Data Ingestion
Start Time: 2025-10-03 06:25:55
Raw Data Path: c:\Users\paulr\desktop\Capstone\data\raw\economy
Processed Data Path: c:\Users\paulr\desktop\Capstone\data\processed
Objective: Process Statistik Austria economic indicators (monthly)
18 economic variables including production indices, trade, employment
Date range: 2009-2025

IMPORTANT ARCHITECTURAL DECISION:
Economic data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate, production, and gas data architecture.
Economic variables can only be used in monthly-level analyses.

DATE FORMAT: mmm.yy (e.g., Jän.09) with Austrian German month abbreviations
COMPLEX STRUCTURE: Multiple header rows and footer metadata require careful skipping.
Cell 1: Configuration and Setup - READY


In [3]:
# =============================================================================
# CELL 2: Missing Values Standardization Function (with Quality Control)
# =============================================================================

def standardize_missing_values(df, additional_missing=None, show_quality_control=True):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
        show_quality_control (bool): Whether to display quality control output
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    if show_quality_control:
        print("\nMISSING VALUES STANDARDIZATION - QUALITY CONTROL:")
        print("-" * 55)
        print("Missing indicators standardized:")
        print("  ", ', '.join([f"'{ind}'" for ind in missing_indicators]))
        print()
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    if show_quality_control:
        if found_patterns:
            print("CONVERSION RESULTS BY COLUMN:")
            for col, pattern in found_patterns.items():
                print(f"  {col}:")
                print(f"    Original nulls: {pattern['original_nulls']}")
                print(f"    Converted missing: {pattern['converted_missing']}")
                print(f"    Total nulls: {pattern['total_nulls']}")
        else:
            print("No missing value patterns found for conversion")
        print("-" * 55)
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function - READY")

Cell 2: Missing Values Standardization Function - READY


In [4]:
# =============================================================================
# CELL 3: Economic Indicators File Discovery
# =============================================================================

def discover_economy_file(data_path, filename):
    """
    Discover single economic indicators data file.
    
    Args:
        data_path (Path): Path to raw data directory
        filename (str): Expected filename
    
    Returns:
        dict: File information dictionary
    """
    file_path = data_path / filename
    
    if file_path.exists():
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': True
        }
    else:
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': False
        }

def display_file_discovery_results(file_info):
    """Display results of economic indicators file discovery."""
    print("Economic indicators file discovery:")
    print(f"  File: {file_info['filename']}")
    print(f"  Exists: {file_info['exists']}")
    print(f"  Path: {file_info['path']}")
    
    if file_info['exists']:
        print("Ready to proceed with economic data loading")
    else:
        print("ERROR: Economic indicators file not found!")

# Execute file discovery
economy_file_info = discover_economy_file(RAW_DATA_PATH, ECON_FILE_NAME)
display_file_discovery_results(economy_file_info)

print("Cell 3: Economic Indicators File Discovery - COMPLETE")

Economic indicators file discovery:
  File: table_2025-09-16_13-30-40.xlsx
  Exists: True
  Path: c:\Users\paulr\desktop\Capstone\data\raw\economy\table_2025-09-16_13-30-40.xlsx
Ready to proceed with economic data loading
Cell 3: Economic Indicators File Discovery - COMPLETE


In [5]:
# =============================================================================
# CELL 4: Load and Clean Economic Indicators Data (REVISED v2)
# =============================================================================

def is_valid_date_format(value):
    """
    Check if value matches Austrian economic date format mmm.yy
    
    Args:
        value: String to check
    
    Returns:
        bool: True if valid date format, False otherwise
    """
    if pd.isna(value) or value == '':
        return False
    try:
        s = str(value).strip()
        # Valid format: 3-4 chars + '.' + 2 chars (e.g., "Jän.09" or "Sept.25")
        if '.' not in s:
            return False
        parts = s.split('.')
        return len(parts) == 2 and len(parts[0]) <= 4 and len(parts[1]) == 2
    except:
        return False

def parse_austrian_economic_date(date_str):
    """
    Parse Austrian date format mmm.yy to datetime.
    Example: 'Jän.09' → 2009-01-01
    
    Assumes all dates are 2000s (09 → 2009, 25 → 2025)
    
    Args:
        date_str (str): Date string in format "Jän.09"
    
    Returns:
        datetime: Parsed datetime object (first day of month)
    """
    if pd.isna(date_str):
        return pd.NaT
    
    try:
        month_abbr, year = str(date_str).strip().split('.')
        month_num = GERMAN_MONTHS.get(month_abbr, month_abbr)
        return pd.to_datetime(f"20{year}-{month_num}-01")
    except Exception as e:
        # Silently return NaT for invalid dates (footer text)
        return pd.NaT

def load_economy_data(file_path, skiprows, column_mapping):
    """
    Load economic indicators data from Excel and select relevant columns.
    Automatically detects end of data by validating date format.
    
    Args:
        file_path (str/Path): Path to economic indicators Excel file
        skiprows (list): Row indices to skip (metadata rows)
        column_mapping (dict): Mapping of source to target column names
    
    Returns:
        tuple: (dataframe, metadata)
    """
    try:
        # Load Excel file without nrows - will filter later
        df = pd.read_excel(file_path, skiprows=skiprows)
        
        print("RAW EXCEL LOAD:")
        print(f"  Initial shape: {df.shape}")
        print()
        
        # Drop first column (empty index column)
        df = df.iloc[:, 1:]
        
        print(f"After dropping column 0: {df.shape}")
        print()
        
        # Strip whitespace from column names
        df.columns = df.columns.str.strip()
        
        # ===================================================================
        # VALIDATION CHECKPOINT 1: Column Detection
        # ===================================================================
        print("COLUMN VALIDATION:")
        print("-" * 60)
        date_col_name = df.columns[0]
        print(f"  First column name: '{date_col_name}'")
        
        # ===================================================================
        # Filter to valid data rows (only rows with valid date format)
        # ===================================================================
        print("\nDATA ROW FILTERING:")
        print("-" * 60)
        first_col = df.columns[0]
        
        # Find valid rows (rows with valid date format mmm.yy)
        valid_rows = df[first_col].apply(is_valid_date_format)
        rows_before = len(df)
        df = df[valid_rows].reset_index(drop=True)
        rows_after = len(df)
        
        print(f"  Rows before filtering: {rows_before}")
        print(f"  Rows after filtering: {rows_after}")
        print(f"  Removed {rows_before - rows_after} footer/invalid rows")
        print()
        
        # ===================================================================
        # VALIDATION CHECKPOINT 2: Data Range
        # ===================================================================
        print("DATA RANGE VALIDATION:")
        print("-" * 60)
        print(f"  First date value (raw): {df.iloc[0, 0]}")
        print(f"  Last date value (raw): {df.iloc[-1, 0]}")
        print(f"  Shape after filtering: {df.shape}")
        print(f"  Expected: (~200, 19)")
        print()
        
        # ===================================================================
        # Parse Austrian dates
        # ===================================================================
        print("DATE PARSING:")
        print("-" * 60)
        
        df[first_col] = df[first_col].apply(parse_austrian_economic_date)
        
        # ===================================================================
        # VALIDATION CHECKPOINT 3: Date Parsing Success
        # ===================================================================
        print(f"  First parsed date: {df[first_col].iloc[0]}")
        print(f"  Last parsed date: {df[first_col].iloc[-1]}")
        print(f"  Date range: {df[first_col].min()} to {df[first_col].max()}")
        failed_parses = df[first_col].isna().sum()
        print(f"  Failed parses: {failed_parses}")
        if failed_parses > 0:
            print(f"  ⚠️ WARNING: {failed_parses} dates could not be parsed!")
        else:
            print(f"  ✓ All dates parsed successfully")
        print()
        
        # ===================================================================
        # VALIDATION CHECKPOINT 4: Column Matching
        # ===================================================================
        print("COLUMN SELECTION VALIDATION:")
        print("-" * 60)
        
        # Adjust column mapping to use actual first column name
        adjusted_mapping = column_mapping.copy()
        if first_col != 'Berichtszeitraum':
            adjusted_mapping[first_col] = adjusted_mapping.pop('Berichtszeitraum')
        
        actual_cols = set(df.columns)
        expected_cols = set(adjusted_mapping.keys())
        missing_cols = expected_cols - actual_cols
        extra_cols = actual_cols - expected_cols
        
        if missing_cols:
            print(f"  ⚠️ Missing columns: {missing_cols}")
        if extra_cols:
            print(f"  ⚠️ Unexpected columns: {extra_cols}")
        if not missing_cols and not extra_cols:
            print(f"  ✓ All {len(expected_cols)} expected columns present")
        print()
        
        # Clean missing values BEFORE renaming
        df_clean, missing_patterns = standardize_missing_values(df, show_quality_control=True)
        
        # Rename columns to standardized names
        df_clean.rename(columns=adjusted_mapping, inplace=True)
        
        print(f"\nCOLUMN RENAMING:")
        print(f"  Renamed {len(adjusted_mapping)} columns with econ_ prefix")
        print()
        
        # Create metadata
        metadata = {
            'filename': Path(file_path).name,
            'rows': len(df_clean),
            'columns': list(df_clean.columns),
            'missing_patterns_found': missing_patterns,
            'date_range': (df_clean['date'].min(), df_clean['date'].max()) if len(df_clean) > 0 else (None, None)
        }
        
        print(f"DATA SUMMARY:")
        print(f"  Final shape: {df_clean.shape}")
        print(f"  Date range: {metadata['date_range'][0].strftime('%Y-%m-%d')} to {metadata['date_range'][1].strftime('%Y-%m-%d')}")
        print()
        
        print("SAMPLE DATA:")
        print(df_clean.head(10))
        
        return df_clean, metadata
        
    except Exception as e:
        print(f"ERROR loading economic data: {e}")
        import traceback
        traceback.print_exc()
        return None, {'error': str(e)}

# Execute data loading
if economy_file_info['exists']:
    economy_df, economy_metadata = load_economy_data(
        economy_file_info['path'],
        SKIPROWS,
        COLUMN_MAPPING
    )
    
    if economy_df is None:
        print(f"ERROR loading economic data: {economy_metadata['error']}")
else:
    print("Skipping data load - file not found")

print("Cell 4: Load and Clean Economic Indicators Data - COMPLETE")

RAW EXCEL LOAD:
  Initial shape: (227, 20)

After dropping column 0: (227, 19)

COLUMN VALIDATION:
------------------------------------------------------------
  First column name: 'Unnamed: 1'

DATA ROW FILTERING:
------------------------------------------------------------
  Rows before filtering: 227
  Rows after filtering: 200
  Removed 27 footer/invalid rows

DATA RANGE VALIDATION:
------------------------------------------------------------
  First date value (raw): Jän.09
  Last date value (raw): Aug.25
  Shape after filtering: (200, 19)
  Expected: (~200, 19)

DATE PARSING:
------------------------------------------------------------
  First parsed date: 2009-01-01 00:00:00
  Last parsed date: 2025-08-01 00:00:00
  Date range: 2009-01-01 00:00:00 to 2025-08-01 00:00:00
  Failed parses: 0
  ✓ All dates parsed successfully

COLUMN SELECTION VALIDATION:
------------------------------------------------------------
  ✓ All 19 expected columns present


MISSING VALUES STANDARDIZATION

In [7]:
# =============================================================================
# CELL 5: Convert Data Types and Create Long Format
# =============================================================================

def convert_economy_data_types(df):
    """
    Convert economic indicator columns to appropriate numeric types.
    Large values (Euro amounts, counts) are rounded to integers (Int64).
    Indices kept as float64 for decimal precision.
    
    Args:
        df (DataFrame): Economy data with econ_ prefixed columns
    
    Returns:
        DataFrame: Data with converted types
    """
    df_converted = df.copy()
    
    print("DATA TYPE CONVERSION:")
    print("-" * 40)
    
    # Get all economy columns (exclude date column)
    econ_columns = [col for col in df_converted.columns if col.startswith('econ_')]
    
    print(f"Converting {len(econ_columns)} economic indicator columns:")
    print()
    
    # Columns that should be rounded to integers (counts, euro amounts in thousands)
    integer_columns = [
        'econ_exports_total_EUR',
        'econ_count_overnight_stays',
        'econ_turnover_industry',
        'econ_count_employees_industry',
        'econ_total_production_construction',
        'econ_turnover_construction',
        'econ_count_employees_construction',
        'econ_imports_total_EUR',
        'econ_imports_energy_EUR'
    ]
    
    # Columns that should keep decimals (indices)
    float_columns = [
        'econ_prod_index_industry',
        'econ_consumer_price_index',
        'econ_turnover_index_commercial_sales',
        'econ_prod_index_construction',
        'econ_turnover_index_construction',
        'econ_turnover_index_wholesale',
        'econ_turnover_index_retail',
        'econ_count_employees_retail',
        'econ_turnover_index_car_retail'
    ]
    
    for col in econ_columns:
        # Convert to numeric first
        df_converted[col] = pd.to_numeric(df_converted[col], errors='coerce')
        
        if col in integer_columns:
            # Round and convert to Int64 (will become float64 in CSV)
            df_converted[col] = df_converted[col].round(0).astype('Int64')
            print(f"  {col}: Int64 (rounded)")
        else:
            # Keep as float64
            df_converted[col] = df_converted[col].astype('float64')
            print(f"  {col}: float64 (decimal)")
    
    print()
    print("NOTE: Int64 types will convert to float64 when saved to CSV")
    print("This is an accepted limitation - float64 is compatible with all analysis tools")
    
    return df_converted

def transform_economy_to_long_format(df, date_column='date'):
    """
    Transform economic data to long format.
    IMPORTANT: Only creates MONTHLY rows (no daily/weekly as per architecture decision).
    
    Args:
        df (DataFrame): Economy data with standardized columns
        date_column (str): Name of date column
    
    Returns:
        DataFrame: Economy data in long format (monthly only)
    """
    print("\nTRANSFORMING TO LONG FORMAT:")
    print("-" * 40)
    print("ARCHITECTURAL NOTE: Creating MONTHLY rows only")
    print("No daily or weekly rows will be created for economic data")
    print("Consistent with climate, production, and gas data architecture")
    print()
    
    df_long = df.copy()
    
    # Ensure date is datetime
    df_long[date_column] = pd.to_datetime(df_long[date_column])
    
    # Add time components
    df_long['year'] = df_long[date_column].dt.year
    df_long['month'] = df_long[date_column].dt.month
    df_long['quarter'] = df_long[date_column].dt.quarter
    df_long['week'] = df_long[date_column].dt.isocalendar().week
    df_long['month_name'] = df_long[date_column].dt.month_name()
    
    # Add aggregation level - MONTHLY ONLY
    df_long['aggregation_level'] = 'monthly'
    
    # Convert date to first of month in ISO format (YYYY-MM-DD)
    df_long['date'] = df_long[date_column].apply(lambda x: x.replace(day=1).strftime('%Y-%m-%d'))
    
    # Select final columns in correct order
    base_columns = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
    econ_columns = [col for col in df_long.columns if col.startswith('econ_')]
    final_columns = base_columns + econ_columns
    
    df_final = df_long[final_columns].copy()
    
    print(f"FINAL STRUCTURE:")
    print(f"  Rows: {len(df_final)} (monthly only)")
    print(f"  Columns: {len(df_final.columns)}")
    print(f"  Economic variables: {len(econ_columns)}")
    print(f"  Date range: {df_final['date'].min()} to {df_final['date'].max()}")
    print()
    
    print("Economic indicator columns:")
    for col in econ_columns:
        print(f"  {col}")
    print()
    
    print("Sample data:")
    print(df_final.head())
    
    return df_final

# Execute data type conversion and transformation
if 'economy_df' in locals() and economy_df is not None:
    economy_converted = convert_economy_data_types(economy_df)
    
    print(f"\nData types after conversion:")
    for col in economy_converted.columns:
        if col.startswith('econ_'):
            print(f"  {col}: {economy_converted[col].dtype}")
    
    economy_long_format = transform_economy_to_long_format(economy_converted)
    
    print("\nLONG FORMAT TRANSFORMATION SUCCESSFUL")
    print(f"  Shape: {economy_long_format.shape}")
else:
    print("Skipping transformation - economy data not available")

print("Cell 5: Convert Data Types and Create Long Format - COMPLETE")

DATA TYPE CONVERSION:
----------------------------------------
Converting 18 economic indicator columns:

  econ_prod_index_industry: float64 (decimal)
  econ_consumer_price_index: float64 (decimal)
  econ_exports_total_EUR: Int64 (rounded)
  econ_turnover_index_commercial_sales: float64 (decimal)
  econ_count_overnight_stays: Int64 (rounded)
  econ_turnover_industry: Int64 (rounded)
  econ_count_employees_industry: Int64 (rounded)
  econ_prod_index_construction: float64 (decimal)
  econ_total_production_construction: Int64 (rounded)
  econ_turnover_index_construction: float64 (decimal)
  econ_turnover_construction: Int64 (rounded)
  econ_count_employees_construction: Int64 (rounded)
  econ_turnover_index_wholesale: float64 (decimal)
  econ_turnover_index_retail: float64 (decimal)
  econ_count_employees_retail: float64 (decimal)
  econ_turnover_index_car_retail: float64 (decimal)
  econ_imports_total_EUR: Int64 (rounded)
  econ_imports_energy_EUR: Int64 (rounded)

NOTE: Int64 types wil

In [8]:
# =============================================================================
# CELL 6: Save Economic Indicators Dataset
# =============================================================================

def save_economy_dataset(df, output_dir, filename="economy_consolidated.csv"):
    """
    Save economic indicators dataset as separate validation sample.
    
    Args:
        df (DataFrame): Final economic indicators dataset
        output_dir (Path): Directory for outputs
        filename (str): Output filename
    
    Returns:
        Path: Path to saved file
    """
    if df is None or len(df) == 0:
        print("No economy data to save!")
        return None
    
    output_path = output_dir / filename
    df.to_csv(output_path, index=False, na_rep='')
    
    print(f"ECONOMIC INDICATORS DATASET SAVED:")
    print(f"  Path: {output_path}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Aggregation level: {df['aggregation_level'].unique()}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    
    # Show economy columns summary
    econ_columns = [col for col in df.columns if col.startswith('econ_')]
    print(f"  Economic variables: {len(econ_columns)}")
    
    return output_path

# Save economy dataset as validation sample
if 'economy_long_format' in locals() and economy_long_format is not None:
    economy_file_path = save_economy_dataset(economy_long_format, PROCESSED_DATA_PATH)
    
    if economy_file_path:
        print("\nECONOMIC INDICATORS FORK SUCCESSFUL")
        print("Ready for merge with data_consolidated.csv")
else:
    print("Skipping save - economy_long_format not available")

print("Cell 6: Save Economic Indicators Dataset - COMPLETE")

ECONOMIC INDICATORS DATASET SAVED:
  Path: c:\Users\paulr\desktop\Capstone\data\processed\economy_consolidated.csv
  Rows: 200
  Columns: 25
  Aggregation level: ['monthly']
  Date range: 2009-01-01 to 2025-08-01
  Economic variables: 18

ECONOMIC INDICATORS FORK SUCCESSFUL
Ready for merge with data_consolidated.csv
Cell 6: Save Economic Indicators Dataset - COMPLETE


In [9]:
# =============================================================================
# CELL 7: Merge Economic Indicators with Consolidated Data
# =============================================================================

def merge_economy_with_consolidated_data(economy_df, consolidated_file_path, output_dir):
    """
    Merge economic indicators data with existing consolidated data.
    
    Args:
        economy_df (DataFrame): Economy dataset in long format (monthly only)
        consolidated_file_path (str/Path): Path to data_consolidated.csv
        output_dir (Path): Directory for final output
    
    Returns:
        DataFrame: Merged dataset with economy data added
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return pd.DataFrame()
    
    print("MERGING ECONOMIC INDICATORS WITH CONSOLIDATED DATA:")
    print("-" * 50)
    
    try:
        # Load existing consolidated data
        consolidated_df = pd.read_csv(consolidated_path)
        
        print(f"Existing consolidated data loaded:")
        print(f"  Shape: {consolidated_df.shape}")
        print(f"  Aggregation levels: {consolidated_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        print(f"Economic data to merge:")
        print(f"  Shape: {economy_df.shape}")
        print(f"  Aggregation levels: {economy_df['aggregation_level'].value_counts().to_dict()}")
        print(f"  Date range: {economy_df['date'].min()} to {economy_df['date'].max()}")
        print()
        
        # Merge on date and aggregation_level
        merged_df = pd.merge(
            consolidated_df, 
            economy_df, 
            on=['date', 'aggregation_level'], 
            how='outer',  # Keep all rows from both datasets
            suffixes=('', '_economy')
        )
        
        print(f"After merge:")
        print(f"  Shape: {merged_df.shape}")
        print()
        
        # Handle duplicate columns from merge
        duplicate_cols = ['year', 'month', 'quarter', 'week', 'month_name']
        for col in duplicate_cols:
            economy_col = f"{col}_economy"
            if economy_col in merged_df.columns:
                merged_df[col] = merged_df[col].fillna(merged_df[economy_col])
                merged_df.drop(economy_col, axis=1, inplace=True)
                print(f"  Resolved duplicate: {col}")
        
        # Sort by date and aggregation level
        merged_df = merged_df.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        
        # Save merged dataset
        final_path = output_dir / "data_consolidated.csv"
        merged_df.to_csv(final_path, index=False, na_rep='')
        
        print(f"\nFINAL CONSOLIDATED DATASET:")
        print(f"  Saved to: {final_path}")
        print(f"  Final shape: {merged_df.shape}")
        print(f"  Date range: {merged_df['date'].min()} to {merged_df['date'].max()}")
        print(f"  Aggregation levels: {merged_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        # Show which columns are economy-specific
        economy_columns = [col for col in merged_df.columns if col.startswith('econ_')]
        print(f"Economic indicator columns added ({len(economy_columns)}):")
        for col in economy_columns:
            non_null = merged_df[col].notna().sum()
            print(f"  {col}: {non_null} non-null values")
        
        return merged_df
        
    except Exception as e:
        print(f"ERROR during merge: {e}")
        return pd.DataFrame()

# Execute merge with consolidated data
if 'economy_long_format' in locals() and economy_long_format is not None:
    consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
    final_merged_df = merge_economy_with_consolidated_data(
        economy_long_format, 
        consolidated_file, 
        PROCESSED_DATA_PATH
    )
    
    if len(final_merged_df) > 0:
        print("\nECONOMIC INDICATORS MERGE SUCCESSFUL")
        print(f"data_consolidated.csv now contains: Electricity + Carbon + Climate + Production + Gas + Economy")
    else:
        print("\nMERGE FAILED")
else:
    print("Skipping merge - economy_long_format not available")

print("Cell 7: Merge Economic Indicators with Consolidated Data - COMPLETE")

MERGING ECONOMIC INDICATORS WITH CONSOLIDATED DATA:
--------------------------------------------------
Existing consolidated data loaded:
  Shape: (4651, 41)
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 189}

Economic data to merge:
  Shape: (200, 25)
  Aggregation levels: {'monthly': 200}
  Date range: 2009-01-01 to 2025-08-01

After merge:
  Shape: (4663, 64)

  Resolved duplicate: year
  Resolved duplicate: month
  Resolved duplicate: quarter
  Resolved duplicate: week
  Resolved duplicate: month_name

FINAL CONSOLIDATED DATASET:
  Saved to: c:\Users\paulr\desktop\Capstone\data\processed\data_consolidated.csv
  Final shape: (4663, 59)
  Date range: 2009-01-01 to 2025-09-01
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 201}

Economic indicator columns added (18):
  econ_prod_index_industry: 55 non-null values
  econ_consumer_price_index: 115 non-null values
  econ_exports_total_EUR: 198 non-null values
  econ_turnover_index_commercial_sales: 54 no

In [10]:
# =============================================================================
# CELL 8: Comprehensive Diagnostics for Final Consolidated Dataset
# =============================================================================

def run_comprehensive_diagnostics(consolidated_file_path):
    """
    Run comprehensive quality control diagnostics on final consolidated dataset.
    Verifies data integrity, completeness, and structure across all data sources.
    
    Args:
        consolidated_file_path (Path): Path to data_consolidated.csv
    
    Returns:
        DataFrame: Loaded consolidated dataset for further inspection
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return None
    
    print("="*70)
    print("COMPREHENSIVE DIAGNOSTICS: data_consolidated.csv")
    print("="*70)
    print()
    
    try:
        df = pd.read_csv(consolidated_path)
        
        # =====================================================================
        # 1. OVERALL STRUCTURE
        # =====================================================================
        print("1. OVERALL STRUCTURE:")
        print("-" * 70)
        print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"  Date range (overall): {df['date'].min()} to {df['date'].max()}")
        print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        print()
        
        # =====================================================================
        # 2. AGGREGATION LEVEL DISTRIBUTION
        # =====================================================================
        print("2. AGGREGATION LEVEL DISTRIBUTION:")
        print("-" * 70)
        agg_counts = df['aggregation_level'].value_counts().sort_index()
        for level, count in agg_counts.items():
            pct = (count / len(df)) * 100
            date_range_level = df[df['aggregation_level'] == level]['date'].agg(['min', 'max'])
            print(f"  {level:10s}: {count:5d} rows ({pct:5.1f}%) | Range: {date_range_level['min']} to {date_range_level['max']}")
        print()
        
        # =====================================================================
        # 3. COLUMN INVENTORY BY DATA SOURCE
        # =====================================================================
        print("3. COLUMN INVENTORY BY DATA SOURCE:")
        print("-" * 70)
        
        base_cols = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
        electricity_cols = [col for col in df.columns if col.startswith('price_') or col == 'data_completeness']
        carbon_cols = [col for col in df.columns if col.startswith('carbonprices_')]
        climate_cols = [col for col in df.columns if col.startswith('climate_')]
        production_cols = [col for col in df.columns if col.startswith('prod_')]
        gas_cols = [col for col in df.columns if col.startswith('oegpi_')]
        economy_cols = [col for col in df.columns if col.startswith('econ_')]
        
        print(f"  Base structure: {len(base_cols)} columns")
        print(f"  Electricity:    {len(electricity_cols)} columns - {electricity_cols}")
        print(f"  Carbon:         {len(carbon_cols)} columns - {carbon_cols}")
        print(f"  Climate:        {len(climate_cols)} columns - {climate_cols}")
        print(f"  Production:     {len(production_cols)} columns")
        if len(production_cols) > 0:
            print(f"    (Showing first 5: {production_cols[:5]})")
        print(f"  Gas Prices:     {len(gas_cols)} columns - {gas_cols}")
        print(f"  Economy:        {len(economy_cols)} columns")
        if len(economy_cols) > 0:
            print(f"    (Showing first 5: {economy_cols[:5]})")
        print(f"  Total: {len(df.columns)} columns")
        print()
        
        # =====================================================================
        # 4. MISSING VALUES ANALYSIS
        # =====================================================================
        print("4. MISSING VALUES ANALYSIS:")
        print("-" * 70)
        
        def analyze_missing_by_source(columns, source_name):
            if len(columns) == 0:
                print(f"  {source_name}: No columns found")
                return
            
            print(f"  {source_name}:")
            for col in columns:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        
        analyze_missing_by_source(electricity_cols, "ELECTRICITY")
        print()
        analyze_missing_by_source(carbon_cols, "CARBON")
        print()
        analyze_missing_by_source(climate_cols, "CLIMATE")
        print()
        
        # Production columns - show all
        if len(production_cols) > 0:
            print(f"  PRODUCTION (summary of {len(production_cols)} columns):")
            for col in production_cols:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        print()
        
        analyze_missing_by_source(gas_cols, "GAS PRICES")
        print()
        
        # Economy columns - show all
        if len(economy_cols) > 0:
            print(f"  ECONOMY (summary of {len(economy_cols)} columns):")
            for col in economy_cols:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        print()
        
        # =====================================================================
        # 5. DATA TYPES VERIFICATION
        # =====================================================================
        print("5. DATA TYPES VERIFICATION:")
        print("-" * 70)
        print("  Expected: All numeric columns should be float64 (due to CSV format)")
        print()
        
        numeric_cols = electricity_cols + carbon_cols + climate_cols + production_cols + gas_cols + economy_cols
        non_float_cols = [col for col in numeric_cols if df[col].dtype != 'float64']
        
        if len(non_float_cols) > 0:
            print(f"  WARNING: {len(non_float_cols)} columns are not float64:")
            for col in non_float_cols:
                print(f"    {col}: {df[col].dtype}")
        else:
            print(f"  ✓ All {len(numeric_cols)} numeric columns are float64")
        print()
        
        # =====================================================================
        # 6. ARCHITECTURAL VALIDATION
        # =====================================================================
        print("6. ARCHITECTURAL VALIDATION:")
        print("-" * 70)
        
        # Check daily rows - should only have electricity + carbon
        daily_df = df[df['aggregation_level'] == 'daily']
        if len(daily_df) > 0:
            daily_climate_nulls = daily_df[climate_cols].isna().all(axis=1).sum()
            daily_prod_nulls = daily_df[production_cols].isna().all(axis=1).sum() if len(production_cols) > 0 else len(daily_df)
            daily_gas_nulls = daily_df[gas_cols].isna().all(axis=1).sum() if len(gas_cols) > 0 else len(daily_df)
            daily_econ_nulls = daily_df[economy_cols].isna().all(axis=1).sum() if len(economy_cols) > 0 else len(daily_df)
            print(f"  Daily rows ({len(daily_df)}):")
            print(f"    Climate columns all null: {daily_climate_nulls}/{len(daily_df)} rows ({daily_climate_nulls/len(daily_df)*100:.1f}%) ✓")
            print(f"    Production columns all null: {daily_prod_nulls}/{len(daily_df)} rows ({daily_prod_nulls/len(daily_df)*100:.1f}%) ✓")
            print(f"    Gas columns all null: {daily_gas_nulls}/{len(daily_df)} rows ({daily_gas_nulls/len(daily_df)*100:.1f}%) ✓")
            print(f"    Economy columns all null: {daily_econ_nulls}/{len(daily_df)} rows ({daily_econ_nulls/len(daily_df)*100:.1f}%) ✓")
        
        # Check weekly rows - should only have electricity + carbon
        weekly_df = df[df['aggregation_level'] == 'weekly']
        if len(weekly_df) > 0:
            weekly_climate_nulls = weekly_df[climate_cols].isna().all(axis=1).sum()
            weekly_prod_nulls = weekly_df[production_cols].isna().all(axis=1).sum() if len(production_cols) > 0 else len(weekly_df)
            weekly_gas_nulls = weekly_df[gas_cols].isna().all(axis=1).sum() if len(gas_cols) > 0 else len(weekly_df)
            weekly_econ_nulls = weekly_df[economy_cols].isna().all(axis=1).sum() if len(economy_cols) > 0 else len(weekly_df)
            print(f"  Weekly rows ({len(weekly_df)}):")
            print(f"    Climate columns all null: {weekly_climate_nulls}/{len(weekly_df)} rows ({weekly_climate_nulls/len(weekly_df)*100:.1f}%) ✓")
            print(f"    Production columns all null: {weekly_prod_nulls}/{len(weekly_df)} rows ({weekly_prod_nulls/len(weekly_df)*100:.1f}%) ✓")
            print(f"    Gas columns all null: {weekly_gas_nulls}/{len(weekly_df)} rows ({weekly_gas_nulls/len(weekly_df)*100:.1f}%) ✓")
            print(f"    Economy columns all null: {weekly_econ_nulls}/{len(weekly_df)} rows ({weekly_econ_nulls/len(weekly_df)*100:.1f}%) ✓")
        
        # Check monthly rows - should have all sources
        monthly_df = df[df['aggregation_level'] == 'monthly']
        if len(monthly_df) > 0:
            monthly_climate_data = monthly_df[climate_cols].notna().any(axis=1).sum()
            monthly_prod_data = monthly_df[production_cols].notna().any(axis=1).sum() if len(production_cols) > 0 else 0
            monthly_gas_data = monthly_df[gas_cols].notna().any(axis=1).sum() if len(gas_cols) > 0 else 0
            monthly_econ_data = monthly_df[economy_cols].notna().any(axis=1).sum() if len(economy_cols) > 0 else 0
            print(f"  Monthly rows ({len(monthly_df)}):")
            print(f"    Rows with climate data: {monthly_climate_data}/{len(monthly_df)} ({monthly_climate_data/len(monthly_df)*100:.1f}%)")
            print(f"    Rows with production data: {monthly_prod_data}/{len(monthly_df)} ({monthly_prod_data/len(monthly_df)*100:.1f}%)")
            print(f"    Rows with gas data: {monthly_gas_data}/{len(monthly_df)} ({monthly_gas_data/len(monthly_df)*100:.1f}%)")
            print(f"    Rows with economy data: {monthly_econ_data}/{len(monthly_df)} ({monthly_econ_data/len(monthly_df)*100:.1f}%)")
        print()
        
        # =====================================================================
        # 7. SAMPLE DATA FROM EACH AGGREGATION LEVEL
        # =====================================================================
        print("7. SAMPLE DATA FROM EACH AGGREGATION LEVEL:")
        print("-" * 70)
        
        for level in ['daily', 'weekly', 'monthly']:
            level_df = df[df['aggregation_level'] == level]
            if len(level_df) > 0:
                print(f"\n  {level.upper()} sample (first 3 rows):")
                # Show subset of columns for readability
                sample_cols = ['date', 'aggregation_level', 'price_exaa_mean', 'carbonprices_primary_market']
                if level == 'monthly':
                    sample_cols.extend(['climate_hdd_at', 'prod_gross_electricity_production', 'oegpi_month'])
                    if len(economy_cols) > 0:
                        sample_cols.append(economy_cols[0])  # Add first economy column if exists
                display_cols = [col for col in sample_cols if col in level_df.columns]
                print(level_df[display_cols].head(3).to_string(index=False))
        
        print()
        print("="*70)
        print("DIAGNOSTICS COMPLETE")
        print("="*70)
        
        return df
        
    except Exception as e:
        print(f"ERROR during diagnostics: {e}")
        import traceback
        traceback.print_exc()
        return None

# Execute comprehensive diagnostics
consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
final_diagnostics_df = run_comprehensive_diagnostics(consolidated_file)

if final_diagnostics_df is not None:
    print("\n✓ data_consolidated.csv is ready for EDA and modeling")
    print(f"  Contains: Electricity + Carbon + Climate + Production + Gas + Economy")
    print(f"  Total: {final_diagnostics_df.shape[0]} rows × {final_diagnostics_df.shape[1]} columns")
else:
    print("\n✗ Diagnostics failed - check errors above")

print("Cell 8: Comprehensive Diagnostics - COMPLETE")

COMPREHENSIVE DIAGNOSTICS: data_consolidated.csv

1. OVERALL STRUCTURE:
----------------------------------------------------------------------
  Shape: 4663 rows × 59 columns
  Date range (overall): 2009-01-01 to 2025-09-01
  Memory usage: 2.74 MB

2. AGGREGATION LEVEL DISTRIBUTION:
----------------------------------------------------------------------
  daily     :  3896 rows ( 83.6%) | Range: 2015-01-01 to 2025-08-31
  monthly   :   201 rows (  4.3%) | Range: 2009-01-01 to 2025-09-01
  weekly    :   566 rows ( 12.1%) | Range: 2015-01-05 to 2025-09-01

3. COLUMN INVENTORY BY DATA SOURCE:
----------------------------------------------------------------------
  Base structure: 7 columns
  Electricity:    5 columns - ['price_exaa_mean', 'price_mc_auction_mean', 'price_count_exaa', 'price_count_mc', 'data_completeness']
  Carbon:         3 columns - ['carbonprices_exchange_rate_eur_usd', 'carbonprices_primary_market', 'carbonprices_secondary_market']
  Climate:        4 columns - ['climat